# Skeleton Extraction by Mesh Contraction (Au et al. 2008)

This notebook is organized into 3 distinct sections:
1. **Part 1: Step-by-Step Component Verification:** Validates every algorithm component individually (Laplacian, contraction, connectivity surgery, embedding refinement).
2. **Part 2: High-Level API:** Evaluates the unified end-to-end  pipeline.
3. **Part 3: Skeleton Refinement Methods:** Evaluates geometric midpoint centering and iterative Laplacian tangential smoothing post-processing.

## Part 1: Step-by-Step Component Verification

In [40]:
# All Imports Consolidated
import sys
sys.path.append("..")

import trimesh
import numpy as np
from pathlib import Path

from animgen.core.models.model import BaseModelClass
from animgen.utils.mesh import (
    triangle_areas,
    vertex_areas,
    compute_cotangent_laplacian,
)
from animgen.rigging.mesh_contraction import (
    contract_mesh,
    connectivity_surgery,
    refine_skeleton_embedding,
    extract_skeleton,
)
from animgen.rigging.refine_skelaton import (
    subdivide_and_center_skeleton,
    refine_and_center_skeleton_iterative,
)
from animgen.renderer.visualizations import (
    visualize_skeleton,
    visualize_skeleton_over_mesh,
)

print("All animgen modules and visualization utilities successfully imported!")

All animgen modules and visualization utilities successfully imported!


In [41]:
# Step 1: Load 3D Mesh Model
model_path = Path("generated_data/test/dec_mesh_Sea_Snake.glb")
if not model_path.exists():
    model_path = Path("../generated_data/test/dec_mesh_Sea_Snake.glb")

model = BaseModelClass(model_path)
mesh = model.mesh
print(f"Loaded mesh with {len(mesh.vertices)} vertices and {len(mesh.faces)} faces.")

Rendering Multiviews...: 100%|██████████| 20/20 [00:03<00:00,  5.80it/s]

Loaded mesh with 6431 vertices and 12862 faces.


In [42]:
# Step 2: Compute Triangle and Vertex Areas
t_areas = triangle_areas(mesh.vertices, mesh.faces)
v_areas = vertex_areas(mesh.vertices, mesh.faces)

print(f"Total surface area of mesh:       {mesh.area:.6f}")
print(f"Sum of computed triangle areas:   {t_areas.sum():.6f}")
print(f"Sum of vertex barycentric areas:  {v_areas.sum():.6f}")
print(f"Mean face area:                   {t_areas.mean():.6f}")

Total surface area of mesh:       0.670597
Sum of computed triangle areas:   0.670597
Sum of vertex barycentric areas:  0.670597
Mean face area:                   0.000052


In [43]:
# Step 3: Compute Discrete Cotangent Laplace-Beltrami Operator
L = compute_cotangent_laplacian(mesh.vertices, mesh.faces)
print("Cotangent Laplacian matrix shape:", L.shape)
print("Number of non-zero entries:", L.nnz)
print("Max sum of rows (should be close to 0):", np.abs(L.sum(axis=1)).max())

Cotangent Laplacian matrix shape: (6431, 6431)
Number of non-zero entries: 45017
Max sum of rows (should be close to 0): 3.552713678800501e-15


In [44]:
# Step 4: Geometry Contraction (Constrained Laplacian Smoothing)
print("Starting geometry contraction...")
contracted_vertices = contract_mesh(mesh.vertices, mesh.faces, max_iters=15, epsilon=1e-6)
print("Geometry contraction completed.")

# Display the contracted zero-volume mesh
contracted_mesh = mesh.copy()
contracted_mesh.vertices = contracted_vertices
print("Displaying the contracted zero-volume mesh...")
contracted_mesh.show()

Starting geometry contraction...
Geometry contraction completed.
Displaying the contracted zero-volume mesh...


In [45]:
# Step 5: Connectivity Surgery (Edge Collapse)
print("Starting connectivity surgery...")
skeletal_nodes, skeletal_edges, parent = connectivity_surgery(
    mesh.faces, contracted_vertices, threshold=0.5, no_1d_collapses=True
)
print(f"Connectivity surgery completed.")
print(f"Remaining active skeletal nodes: {len(skeletal_nodes)}")
print(f"Remaining skeletal edges:       {len(skeletal_edges)}")

Starting connectivity surgery...
Connectivity surgery completed.
Remaining active skeletal nodes: 20
Remaining skeletal edges:       19


In [46]:
# Step 6: Embedding Refinement (Boundary Loop Centering)
print("Starting embedding refinement...")
refined_positions = refine_skeleton_embedding(
    mesh.vertices, contracted_vertices, skeletal_nodes, skeletal_edges, parent, mesh.faces
)
print("Embedding refinement completed.")

Starting embedding refinement...
Embedding refinement completed.


In [47]:
# Step 7: Reconstruct and Display 1D Skeleton Graph overlaid on original mesh
node_to_idx = {node: idx for idx, node in enumerate(skeletal_nodes)}
skeleton_vertices = np.array([refined_positions[node] for node in skeletal_nodes])
skeleton_edges_idx = np.array([[node_to_idx[u], node_to_idx[v]] for u, v in skeletal_edges])

print("Creating overlaid visualization scene (low opacity mesh + solid skeleton)...")
scene = visualize_skeleton_over_mesh(model, (skeleton_vertices, skeleton_edges_idx), radius=0.015, opacity=0.25)
print("Displaying the overlay scene (red bones, blue joints, 25% opacity mesh)...")
scene.show()

Creating overlaid visualization scene (low opacity mesh + solid skeleton)...
Displaying the overlay scene (red bones, blue joints, 25% opacity mesh)...


## Part 2: High-Level End-to-End Skeleton Extraction API

In [48]:
# Step 8: Test High-Level extract_skeleton API directly
print("Running top-level extract_skeleton API...")
skeleton = extract_skeleton(mesh, max_iters=20, threshold=0.5, no_1d_collapses=True)
print("Skeleton successfully extracted.")

print("Displaying high-level skeleton overlaid on model...")
scene = visualize_skeleton_over_mesh(model, skeleton, radius=0.015, opacity=0.25)
scene.show()

Running top-level extract_skeleton API...
Skeleton successfully extracted.
Displaying high-level skeleton overlaid on model...


## Part 3: Skeleton Refinement & Centering Post-Processing Methods

In [49]:
# Method A: Geometric Edge Subdivision & Cross-Section Centering
print("Subdividing long skeleton edges and centering midpoints inside mesh cross-sections...")
dense_v_a, dense_e_a = subdivide_and_center_skeleton(
    mesh.vertices, skeleton_vertices, skeleton_edges_idx, max_edge_len=0.03
)
print(f"Generated dense, centered skeleton with {len(dense_v_a)} nodes and {len(dense_e_a)} edges.")

# Display geometrically centered dense skeleton overlaid on original mesh
scene_a = visualize_skeleton_over_mesh(model, (dense_v_a, dense_e_a), radius=0.012, opacity=0.25)
print("Displaying Method A refined skeleton...")
scene_a.show()

Subdividing long skeleton edges and centering midpoints inside mesh cross-sections...
Generated dense, centered skeleton with 192 nodes and 191 edges.
Displaying Method A refined skeleton...


In [50]:
# Method B: Iterative Subdivision + Tangential Laplacian Smoothing + Cross-Section Centering
print("Applying iterative Laplacian smoothing and tangential centering...")
dense_v_b, dense_e_b = refine_and_center_skeleton_iterative(
    mesh.vertices,
    skeleton_vertices,
    skeleton_edges_idx,
    max_edge_len=0.03,
    num_iters=10,
    alpha=0.3,
    beta=0.5,
    laplacian_weight=0.3,
)
print(f"Generated Laplacian-refined skeleton with {len(dense_v_b)} nodes and {len(dense_e_b)} edges.")

# Display Laplacian-refined skeleton overlaid on original mesh
scene_b = visualize_skeleton_over_mesh(model, (dense_v_b, dense_e_b), radius=0.012, opacity=0.25)
print("Displaying Method B refined skeleton...")
scene_b.show()

Applying iterative Laplacian smoothing and tangential centering...
Generated Laplacian-refined skeleton with 207 nodes and 206 edges.
Displaying Method B refined skeleton...
